# APARCH e Component-GARCH

Neste notebook, exploramos dois modelos GARCH avancados:

- **APARCH** (Asymmetric Power ARCH) de Ding, Granger e Engle (1993)
- **IGARCH** (Integrated GARCH) de Engle e Bollerslev (1986)
- **Component-GARCH** de Engle e Lee (1999)

Esses modelos generalizam o GARCH padrao em direcoes diferentes:
o APARCH introduz um **parametro de potencia** flexivel, o IGARCH impoe
**persistencia unitaria**, e o Component-GARCH decompoe a volatilidade em
**componentes de curto e longo prazo**.

**Conteudo:**
1. APARCH de Ding, Granger e Engle (1993)
2. O parametro de potencia delta
3. IGARCH - Integrated GARCH
4. Component-GARCH de Engle e Lee (1999)
5. Componente transitorio vs permanente
6. Aplicacao: dados do Bitcoin

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from archbox.models import APARCH, GARCH, IGARCH, ComponentGARCH

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. APARCH de Ding, Granger e Engle (1993)

O modelo **APARCH** (Asymmetric Power ARCH) e uma generalizacao poderosa que unifica
varios modelos GARCH atraves de um **parametro de potencia** $\delta$:

$$\sigma_t^{\delta} = \omega + \alpha (|\epsilon_{t-1}| - \gamma \epsilon_{t-1})^{\delta} + \beta \sigma_{t-1}^{\delta}$$

onde:
- $\delta > 0$: parametro de potencia (estimado ou fixo)
- $\gamma \in (-1, 1)$: parametro de assimetria (leverage)
- Se $\gamma > 0$: choques negativos tem maior impacto

**Casos especiais:**
| $\delta$ | $\gamma$ | Modelo equivalente |
|----------|----------|--------------------|
| 2 | 0 | GARCH |
| 2 | $\neq 0$ | GJR-GARCH |
| 1 | 0 | AVGARCH (Taylor, 1986) |
| 1 | $\neq 0$ | TARCH (Zakoian, 1994) |

In [ ]:
# Load S&P 500 data
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

# Estimate APARCH(1,1)
model_aparch = APARCH(returns.values, p=1, q=1)
results_aparch = model_aparch.fit()
print(results_aparch.summary())

# Extract delta and gamma
delta_idx = results_aparch.param_names.index('delta')
delta_val = results_aparch.params[delta_idx]
gamma_idx = [i for i, n in enumerate(results_aparch.param_names) if 'gamma' in n][0]
gamma_val = results_aparch.params[gamma_idx]

print(f"\nEstimated delta = {delta_val:.4f}")
print(f"Estimated gamma = {gamma_val:.4f}")
print("\nDelta = 2.0 would be equivalent to GARCH, delta = 1.0 to AVGARCH.")
print(f"The data-driven optimal power is {delta_val:.4f}.")

## 2. O parametro de potencia $\delta$

Ding, Granger e Engle (1993) mostraram empiricamente que a autocorrelacao de $|r_t|^d$
e **maximizada** quando $d \approx 1$ (nao $d = 2$ como no GARCH padrao).

Isso sugere que modelar a **volatilidade** (desvio padrao, $\delta = 1$) pode ser mais
apropriado do que modelar a **variancia** ($\delta = 2$).

O APARCH permite que os dados determinem o valor otimo de $\delta$:
- $\delta = 2$: equivale ao GARCH padrao (modela variancia)
- $\delta = 1$: modela o desvio padrao (AVGARCH)
- $\delta$ livre: o valor otimo e determinado pelos dados

Vamos comparar modelos com diferentes valores fixos de $\delta$.

In [ ]:
# Compare APARCH with different fixed delta values
# APARCH with free delta was already estimated above

# For fixed-delta comparisons, we estimate standard GARCH (delta=2) and
# re-estimate APARCH letting the optimizer find the optimum
model_garch = GARCH(returns.values, p=1, q=1)
results_garch = model_garch.fit(disp=False)

# Collect results for different effective delta values
delta_comparison = pd.DataFrame({
    'GARCH (delta=2)': {
        'AIC': results_garch.aic,
        'BIC': results_garch.bic,
        'LogLik': results_garch.loglike,
        'Params': len(results_garch.params),
        'Persistence': results_garch.persistence(),
    },
    f'APARCH (delta={delta_val:.2f})': {
        'AIC': results_aparch.aic,
        'BIC': results_aparch.bic,
        'LogLik': results_aparch.loglike,
        'Params': len(results_aparch.params),
        'Persistence': results_aparch.persistence(),
    },
})

print("Comparison: GARCH (delta=2) vs APARCH (free delta)")
print("=" * 60)
print(delta_comparison.T.to_string())

best_aic = delta_comparison.loc['AIC'].astype(float).idxmin()
print(f"\nBest by AIC: {best_aic}")
print(f"Estimated delta = {delta_val:.4f}")
if abs(delta_val - 1.0) < abs(delta_val - 2.0):
    print("Delta is closer to 1 (modeling volatility rather than variance).")
else:
    print("Delta is closer to 2 (standard variance modeling).")

# Visualize conditional volatility comparison
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_garch.conditional_volatility, linewidth=0.8, alpha=0.7, label='GARCH(1,1)')
ax.plot(results_aparch.conditional_volatility, linewidth=0.8, alpha=0.7, label=f'APARCH (delta={delta_val:.2f})')
ax.set_title('Conditional Volatility: GARCH vs APARCH')
ax.set_xlabel('Observation')
ax.set_ylabel('Volatility')
ax.legend()
fig.tight_layout()
plt.show()

## 3. IGARCH - Integrated GARCH

O **IGARCH** (Integrated GARCH) de Engle e Bollerslev (1986) e um caso especial
do GARCH(1,1) onde a **persistencia e unitaria**:

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + (1 - \alpha) \sigma_{t-1}^2$$

ou seja, $\alpha + \beta = 1$ (com $\beta = 1 - \alpha$).

**Implicacoes:**
- A variancia incondicional **nao existe** (nao e finita)
- Choques na volatilidade tem efeito **permanente** (nunca se dissipam completamente)
- A previsao de longo prazo **nao converge** para um nivel fixo
- Comum em dados financeiros de alta frequencia

O IGARCH e util quando o GARCH estimado tem persistencia muito proxima de 1.

In [ ]:
# Estimate IGARCH(1,1) with unit root constraint alpha + beta = 1
model_igarch = IGARCH(returns.values)
results_igarch = model_igarch.fit()
print(results_igarch.summary())

# Verify persistence
print(f"\nPersistence: {results_igarch.persistence():.6f}")
print("(Should be 1.0 or very close to 1.0)")

# Compare GARCH vs IGARCH conditional volatility
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(results_garch.conditional_volatility, linewidth=0.8, color='steelblue', label='GARCH(1,1)')
axes[0].set_title('GARCH(1,1) - Conditional Volatility')
axes[0].set_ylabel('Volatility')
axes[0].legend()

axes[1].plot(results_igarch.conditional_volatility, linewidth=0.8, color='darkorange', label='IGARCH(1,1)')
axes[1].set_title('IGARCH(1,1) - Conditional Volatility (unit persistence)')
axes[1].set_ylabel('Volatility')
axes[1].set_xlabel('Observation')
axes[1].legend()

fig.tight_layout()
plt.show()

# Compare information criteria
print("\nGARCH vs IGARCH Comparison")
print("=" * 40)
print(f"{'':15} {'GARCH':>12} {'IGARCH':>12}")
print(f"{'AIC':<15} {results_garch.aic:>12.4f} {results_igarch.aic:>12.4f}")
print(f"{'BIC':<15} {results_garch.bic:>12.4f} {results_igarch.bic:>12.4f}")
print(f"{'Persistence':<15} {results_garch.persistence():>12.6f} {results_igarch.persistence():>12.6f}")

## 4. Component-GARCH de Engle e Lee (1999)

O **Component-GARCH** (CGARCH) decompoe a variancia condicional em dois componentes:

**Componente permanente (tendencia de longo prazo):**
$$q_t = \omega + \rho (q_{t-1} - \omega) + \phi (\epsilon_{t-1}^2 - \sigma_{t-1}^2)$$

**Componente transitorio (desvios de curto prazo):**
$$\sigma_t^2 - q_t = \alpha (\epsilon_{t-1}^2 - q_{t-1}) + \beta (\sigma_{t-1}^2 - q_{t-1})$$

**Variancia condicional total:**
$$\sigma_t^2 = q_t + (\text{componente transitorio})$$

**Interpretacao:**
- $q_t$: nivel de **longo prazo** da volatilidade (varia lentamente)
- $\sigma_t^2 - q_t$: desvio de **curto prazo** (decai rapidamente)
- $\rho$: persistencia do componente permanente (proximo de 1)
- $\alpha + \beta$: persistencia do componente transitorio (menor que $\rho$)

In [ ]:
# Estimate Component-GARCH
model_cgarch = ComponentGARCH(returns.values)
results_cgarch = model_cgarch.fit()
print(results_cgarch.summary())

# Display parameter interpretation
print("\nComponent-GARCH Parameters:")
print("=" * 50)
for name, param in zip(results_cgarch.param_names, results_cgarch.params, strict=False):
    print(f"  {name:<15} = {param:.6f}")

print(f"\nTransitory persistence (alpha + beta): {results_cgarch.params[1] + results_cgarch.params[2]:.6f}")
print(f"Permanent persistence (beta_p):        {results_cgarch.params[4]:.6f}")
print("The permanent component should have higher persistence than the transitory one.")

## 5. Componente transitorio vs permanente

A decomposicao em componentes oferece **interpretacao economica** rica:

- O **componente permanente** ($q_t$) captura mudancas estruturais no nivel de risco
  do mercado (crises, mudancas de regime, tendencias macroeconomicas)
- O **componente transitorio** ($\sigma_t^2 - q_t$) captura choques de curto prazo
  (noticias, eventos, etc.) que se dissipam rapidamente

Essa decomposicao e util para:
- Distinguir entre risco **sistematico** (permanente) e **idiossincratico** (transitorio)
- Definir horizontes de investimento: para horizontes longos, foque no componente permanente
- Detectar **mudancas de regime** no nivel base de volatilidade

In [ ]:
# Plot permanent and transitory volatility components

# Get variance decomposition using the model's method
backcast = np.var(returns.values)
sigma2, q_t, h_t = model_cgarch.variance_decomposition(
    results_cgarch.params, returns.values, backcast
)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Panel 1: Total conditional volatility
axes[0].plot(np.sqrt(sigma2), linewidth=0.8, color='steelblue')
axes[0].set_title('Total Conditional Volatility')
axes[0].set_ylabel('Volatility')

# Panel 2: Permanent component (long-run trend)
axes[1].plot(np.sqrt(np.maximum(q_t, 0)), linewidth=0.8, color='darkred')
axes[1].set_title('Permanent Component (Long-run Trend)')
axes[1].set_ylabel('Volatility')

# Panel 3: Transitory component (short-run deviations)
axes[2].plot(h_t, linewidth=0.8, color='darkorange')
axes[2].set_title('Transitory Component (Short-run Deviations)')
axes[2].set_ylabel('Variance contribution')
axes[2].set_xlabel('Observation')
axes[2].axhline(0, color='grey', linewidth=0.5, linestyle='--')

fig.suptitle('Component-GARCH Variance Decomposition', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

print("The permanent component varies slowly, capturing structural changes in risk.")
print("The transitory component fluctuates rapidly around zero, capturing short-lived shocks.")

## 6. Aplicacao: dados do Bitcoin

O **Bitcoin** apresenta caracteristicas unicas em relacao a ativos tradicionais:

- Volatilidade **muito mais alta** que acoes
- Possiveis **mudancas de regime** na volatilidade (ciclos de bull/bear)
- Caudas **extremamente pesadas**
- Efeito alavancagem potencialmente **diferente** (ou ausente)

O APARCH (com potencia flexivel) e o Component-GARCH (decomposicao em tendencia + ciclo)
sao particularmente interessantes para criptomoedas.

In [ ]:
# Application: Bitcoin data
btc = pd.read_csv('../data/bitcoin_returns.csv', parse_dates=['date'], index_col='date')
btc_returns = btc['returns']

print(f"Bitcoin dataset: {len(btc_returns)} observations")
print(f"Period: {btc_returns.index[0].date()} to {btc_returns.index[-1].date()}")
print(f"Mean: {btc_returns.mean():.6f}, Std: {btc_returns.std():.6f}")
print(f"Skewness: {btc_returns.skew():.4f}, Kurtosis: {btc_returns.kurtosis():.4f}\n")

# Estimate APARCH on Bitcoin
model_aparch_btc = APARCH(btc_returns.values, p=1, q=1)
results_aparch_btc = model_aparch_btc.fit(disp=False)
print("APARCH - Bitcoin")
print(results_aparch_btc.summary())

# Estimate Component-GARCH on Bitcoin
model_cgarch_btc = ComponentGARCH(btc_returns.values)
results_cgarch_btc = model_cgarch_btc.fit(disp=False)
print("\nComponent-GARCH - Bitcoin")
print(results_cgarch_btc.summary())

# Compare delta estimates
delta_btc = results_aparch_btc.params[results_aparch_btc.param_names.index('delta')]
print("\nDelta comparison:")
print(f"  S&P 500:  {delta_val:.4f}")
print(f"  Bitcoin:  {delta_btc:.4f}")

# Information criteria comparison
btc_comparison = pd.DataFrame({
    'APARCH (S&P500)': {'AIC': results_aparch.aic, 'BIC': results_aparch.bic},
    'APARCH (Bitcoin)': {'AIC': results_aparch_btc.aic, 'BIC': results_aparch_btc.bic},
    'CGARCH (S&P500)': {'AIC': results_cgarch.aic, 'BIC': results_cgarch.bic},
    'CGARCH (Bitcoin)': {'AIC': results_cgarch_btc.aic, 'BIC': results_cgarch_btc.bic},
})
print("\nInformation Criteria: S&P 500 vs Bitcoin")
print("=" * 60)
print(btc_comparison.T.to_string())

# Variance decomposition for Bitcoin
backcast_btc = np.var(btc_returns.values)
sigma2_btc, q_t_btc, h_t_btc = model_cgarch_btc.variance_decomposition(
    results_cgarch_btc.params, btc_returns.values, backcast_btc
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# S&P 500 components
axes[0, 0].plot(np.sqrt(np.maximum(q_t, 0)), linewidth=0.8, color='darkred')
axes[0, 0].set_title('S&P 500 - Permanent Component')
axes[0, 0].set_ylabel('Volatility')

axes[1, 0].plot(h_t, linewidth=0.8, color='darkorange')
axes[1, 0].set_title('S&P 500 - Transitory Component')
axes[1, 0].set_ylabel('Variance')

# Bitcoin components
axes[0, 1].plot(np.sqrt(np.maximum(q_t_btc, 0)), linewidth=0.8, color='darkred')
axes[0, 1].set_title('Bitcoin - Permanent Component')
axes[0, 1].set_ylabel('Volatility')

axes[1, 1].plot(h_t_btc, linewidth=0.8, color='darkorange')
axes[1, 1].set_title('Bitcoin - Transitory Component')
axes[1, 1].set_ylabel('Variance')

fig.suptitle('Component-GARCH Decomposition: S&P 500 vs Bitcoin', fontsize=14)
fig.tight_layout()
plt.show()

## Conclusao

Neste notebook, aprendemos:

- O modelo **APARCH** e sua generalizacao via parametro de potencia $\delta$
- Que $\delta \approx 1$ frequentemente ajusta melhor que $\delta = 2$ (GARCH padrao)
- O **IGARCH** e suas implicacoes de persistencia unitaria
- O **Component-GARCH** e a decomposicao em tendencia de longo prazo + ciclo de curto prazo
- Aplicacao pratica em dados de criptomoedas

No proximo notebook, faremos uma **comparacao sistematica** de todos os modelos
GARCH univariados, incluindo diagnosticos, previsao out-of-sample e ranking final.